In [0]:
# Make sure everything is where we expect it
base_path = "/Volumes/finalcomp/default/caretocompare/CARE_To_Compare/CARE_To_Compare"

for item in dbutils.fs.ls(base_path):
    print(item.name)

In [0]:
farm_a_path = f"{base_path}/Wind Farm A"

for item in dbutils.fs.ls(farm_a_path):
    print(item.name, f"({item.size / 1e3:.1f} KB)")

In [0]:
import pandas as pd

farm_a_path = f"{base_path}/Wind Farm A"

event_info = pd.read_csv(f"{farm_a_path}/event_info.csv", sep=";")
display(event_info.head(10))
print(f"\nShape: {event_info.shape}")
print(f"Anomaly events: {(event_info['event_label'] == 'anomaly').sum()}")
print(f"Normal events:  {(event_info['event_label'] == 'normal').sum()}")

In [0]:
feature_desc = pd.read_csv(f"{farm_a_path}/feature_description.csv", sep=";")
display(feature_desc.head(20))
print(f"\nTotal features: {len(feature_desc)}")

In [0]:
import os

datasets_path = f"{farm_a_path}/datasets"

all_files = os.listdir(datasets_path)
csv_files = [f for f in all_files if f.endswith(".csv")]
print(f"Found {len(csv_files)} dataset files")

dfs = []
for fname in csv_files:
    fpath = os.path.join(datasets_path, fname)
    df = pd.read_csv(fpath, sep=";")
    event_id = fname.replace(".csv", "")
    df["event_id"] = event_id
    dfs.append(df)

combined = pd.concat(dfs, ignore_index=True)
print(f"Total rows loaded: {len(combined):,}")
print(f"Columns: {list(combined.columns)}")

In [0]:
# Make sure both event_id columns are the same type before joining
combined["event_id"] = combined["event_id"].astype(int)

# Merge the anomaly/normal label from event_info onto every row
combined = combined.merge(
    event_info[["event_id", "event_label", "event_description"]],
    on="event_id",
    how="left"
)

# Quick check
print(combined["event_label"].value_counts())

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:139)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:139)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:724)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:442)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:442)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

In [0]:
display(combined.head(10))

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:141)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:724)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:442)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:442)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:495)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:725)
	at com.data

In [0]:
spark_df = spark.createDataFrame(combined)

(spark_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("finalcomp.default.farm_a_bronze")
)

print("Done. Row count:")
spark.sql("SELECT COUNT(*) FROM finalcomp.default.farm_a_bronze").show()

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:141)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:724)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:442)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:442)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:495)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:725)
	at com.data

In [0]:
#load farm b
farm_b_path = f"{base_path}/Wind Farm B"

# Load event info
event_info_b = pd.read_csv(f"{farm_b_path}/event_info.csv", sep=";")

# Load all CSVs
datasets_path_b = f"{farm_b_path}/datasets"
csv_files_b = [f for f in os.listdir(datasets_path_b) if f.endswith(".csv")]
print(f"Farm B: found {len(csv_files_b)} files")

dfs_b = []
for fname in csv_files_b:
    df = pd.read_csv(os.path.join(datasets_path_b, fname), sep=";")
    df["event_id"] = int(fname.replace(".csv", ""))
    dfs_b.append(df)

combined_b = pd.concat(dfs_b, ignore_index=True)

# Join labels
combined_b["event_id"] = combined_b["event_id"].astype(int)
combined_b = combined_b.merge(
    event_info_b[["event_id", "event_label", "event_description"]],
    on="event_id",
    how="left"
)

print(f"Total rows: {len(combined_b):,}")
print(combined_b["event_label"].value_counts())

In [0]:
temp_path = f"{base_path}/farm_b_temp.parquet"

combined_b.to_parquet(temp_path, index=False)

(spark.read.parquet(temp_path)
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("finalcomp.default.farm_b_bronze")
)

# Clean up the temp file after writing
dbutils.fs.rm(temp_path)

print("Done. Row count:")
spark.sql("SELECT COUNT(*) FROM finalcomp.default.farm_b_bronze").show()

In [0]:
spark.sql("SHOW TABLES IN finalcomp.default").show(truncate=False)